In [ ]:
import numpy as np
import os
import pandas as pd
import re
from pathlib import Path
import pyreadr

In [ ]:
DATA_DIR = Path("/store_new/mch/msclim/antoumos/R/develop/NOWPRECIP/new_project/radar_data")

In [ ]:
# ---- Load pre-built metadata instead of rebuilding ----
meta_all = pd.read_csv(DATA_DIR / "phase2_samples_meta.csv", parse_dates=["event_datetime", "t0_datetime", "target_datetime"])
print("Loaded meta_all:", meta_all.shape)
print("Unique events:", meta_all["event_id"].nunique())

# ---- Load arrays (once you save them) ----
data = np.load(DATA_DIR / "phase2_samples.npz")
X_all = data["X"]  # (8602, 3, 501, 371) float32
Y_all = data["Y"]  # (8602, 1, 501, 371) float32
print("X_all:", X_all.shape)
print("Y_all:", Y_all.shape)

In [6]:
# derive stats ----
Y_flat = Y_all.reshape(Y_all.shape[0], -1)

meta_all["mean_intensity"] = Y_flat.mean(axis=1)
meta_all["max_intensity"]  = Y_flat.max(axis=1)

wet_mask = Y_flat > 0.1
meta_all["mean_intensity_wet"] = [
    Y_flat[i][wet_mask[i]].mean() if wet_mask[i].any() else 0.0
    for i in range(Y_flat.shape[0])
]
meta_all["wet_fraction"] = wet_mask.sum(axis=1) / Y_flat.shape[1]

# ---- build split ----
events = (
    meta_all[["event_id", "event_datetime", "season"]]
    .drop_duplicates()
    .sort_values(["season", "event_datetime"])
)


In [7]:
### Define operational split ###

train_events, val_events, test_events = [], [], []
for season, df in events.groupby("season"):
    df = df.sort_values("event_datetime")
    n = len(df)
    train_end = int(0.70 * n)
    val_end   = int(0.85 * n)
    train_events.extend(df.iloc[:train_end]["event_id"])
    val_events.extend(df.iloc[train_end:val_end]["event_id"])
    test_events.extend(df.iloc[val_end:]["event_id"])

meta_all["split"] = "train"
meta_all.loc[meta_all["event_id"].isin(val_events),  "split"] = "val"
meta_all.loc[meta_all["event_id"].isin(test_events), "split"] = "test"

print(meta_all["split"].value_counts())

split
train    5984
test     1360
val      1258
Name: count, dtype: int64


In [8]:
# ---- Log transform ----
X_all_log = np.log1p(X_all).astype(np.float32)
Y_all_log = np.log1p(Y_all).astype(np.float32)

# ---- Final splits ----
train_mask = meta_all["split"] == "train"
val_mask   = meta_all["split"] == "val"
test_mask  = meta_all["split"] == "test"

X_train, Y_train = X_all_log[train_mask], Y_all_log[train_mask]
X_val,   Y_val   = X_all_log[val_mask],   Y_all_log[val_mask]
X_test,  Y_test  = X_all_log[test_mask],  Y_all_log[test_mask]

meta_train = meta_all.loc[train_mask].reset_index(drop=True)
meta_val   = meta_all.loc[val_mask].reset_index(drop=True)
meta_test  = meta_all.loc[test_mask].reset_index(drop=True)

print(f"X_train: {X_train.shape}  Y_train: {Y_train.shape}")
print(f"X_val:   {X_val.shape}  Y_val:   {Y_val.shape}")
print(f"X_test:  {X_test.shape}  Y_test:  {Y_test.shape}")


X_train: (5984, 3, 501, 371)  Y_train: (5984, 1, 501, 371)
X_val:   (1258, 3, 501, 371)  Y_val:   (1258, 1, 501, 371)
X_test:  (1360, 3, 501, 371)  Y_test:  (1360, 1, 501, 371)


In [9]:
# ---- Persistence baseline (in original space) ----
Y_persist = X_all[test_mask][:, -1:, :, :]   # shape (N_test, 1, H, W)
mse_persist = float(np.mean((Y_persist - Y_all[test_mask])**2))
mae_persist = float(np.mean(np.abs(Y_persist - Y_all[test_mask])))
print(f"Persistence  MSE: {mse_persist:.6f}  MAE: {mae_persist:.6f}")


Persistence  MSE: 0.237859  MAE: 0.082429


In [10]:
pd.DataFrame([{"model": "persistence", "mse": mse_persist, "mae": mae_persist}])\
  .to_csv(DATA_DIR / "results.csv", index=False)


In [11]:
### Save train/val/test split ###
meta_all.to_csv(DATA_DIR / "phase2_samples_meta_enriched.csv", index=False)
print("Saved enriched metadata with split column.")

# Then in training you just do:
# meta = pd.read_csv("phase2_samples_meta_enriched.csv")
# train_mask = meta["split"] == "train"
# X_train = X_all_log[train_mask]  # fast, no extra disk space

Saved enriched metadata with split column.
